# 📊 Customer Churn Prediction

## 1. Business Understanding

### Objective

Customer churn refers to customers who stop using a company's services. In the telecom industry, acquiring new customers is significantly more expensive than retaining existing ones. Identifying customers who are likely to churn enables businesses to take proactive retention measures, reduce revenue loss, and improve customer satisfaction.

### Business Problem

The telecom company experiences customer attrition, which impacts recurring revenue and long-term profitability. The objective is to identify customers who are at high risk of churning so that targeted retention strategies can be implemented.

### Machine Learning Objective

Develop a binary classification model that predicts whether a customer is likely to churn based on demographic, account, and service-related information.

### Target Variable

* **Target Column:** `Churn`
* **Yes** → Customer churned
* **No** → Customer retained

### Success Criteria

Since losing an existing customer is generally more costly than contacting a customer who would have stayed, the project prioritizes **Recall** while maintaining a balanced overall model performance using Precision, F1-Score, and ROC-AUC.

### Expected Business Impact

An accurate churn prediction model can help the company:

* Identify customers at high risk of leaving.
* Improve customer retention through targeted offers.
* Reduce revenue loss caused by customer attrition.
* Support data-driven decision-making for marketing and customer success teams.


In [1]:

# ================================================================
# INSTALL, IMPORT & VERIFY EDA / MACHINE LEARNING LIBRARIES
# ================================================================

import sys
import subprocess

# Step 1: Install required libraries in the current Python environment
packages = [
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "xgboost",
    "shap",
    "joblib",
]

print("Installing required libraries...\n")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", *packages
])

print("\nAll package installations completed!")

# Step 2: Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
import xgboost
import shap
import joblib

# Step 3: Verify installation
print("\n" + "=" * 50)
print("LIBRARY VERIFICATION")
print("=" * 50)

print("Python executable:", sys.executable)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib:", plt.matplotlib.__version__)
print("Seaborn:", sns.__version__)
print("Scikit-learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)
print("SHAP:", shap.__version__)
print("Joblib:", joblib.__version__)

# Step 4: Test basic functionality
test_df = pd.DataFrame({
    "Age": [20, 25, 30],
    "Salary": [20000, 30000, 40000],
})

assert test_df.shape == (3, 2)
assert np.mean(test_df["Age"]) == 25

print("\nDataFrame test: PASSED")
print("NumPy test: PASSED")
print("All imports successful!")


Installing required libraries...


All package installations completed!


C:\Users\nisha\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



LIBRARY VERIFICATION
Python executable: C:\Users\nisha\AppData\Local\Programs\Python\Python312\python.exe
NumPy: 2.5.4
Pandas: 2.2.3
Matplotlib: 3.11.2
Seaborn: 0.13.2
Scikit-learn: 1.9.1
XGBoost: 3.4.1
SHAP: 0.54.1
Joblib: 1.6.0

DataFrame test: PASSED
NumPy test: PASSED
All imports successful!


In [14]:
churn_df = pd.read_csv(
    r"D:\ML Project\data\WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

In [15]:
churn_df.shape

(7043, 21)

In [16]:
churn_df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [17]:
RANDOM_STATE = 42

display(churn_df.sample(5, random_state=RANDOM_STATE))

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
185,1024-GUALD,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,24.80,24.8,Yes
2715,0484-JPBRU,Male,0,No,No,41,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Bank transfer (automatic),25.25,996.45,No
3825,3620-EHIMZ,Female,0,Yes,Yes,52,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.35,1031.7,No
1807,6910-HADCM,Female,0,No,No,1,Yes,No,Fiber optic,No,...,Yes,No,No,No,Month-to-month,No,Electronic check,76.35,76.35,Yes
132,8587-XYZSF,Male,0,No,No,67,Yes,No,DSL,No,...,No,Yes,No,No,Two year,No,Bank transfer (automatic),50.55,3260.1,No


In [18]:
pd.DataFrame({
    "columns" : churn_df.columns
})

,columns
0,customerID
1,gender
2,SeniorCitizen
3,Partner
4,Dependents
5,tenure
6,PhoneService
7,MultipleLines
8,InternetService
9,OnlineSecurity


In [19]:
churn_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [20]:
unique_values = {"unique_val_counts" : churn_df.nunique()}

pd.DataFrame(unique_values).sort_values(by="unique_val_counts" )

,unique_val_counts
gender,2
SeniorCitizen,2
Partner,2
Dependents,2
PhoneService,2
PaperlessBilling,2
Churn,2
MultipleLines,3
TechSupport,3
StreamingTV,3


In [21]:
numerical_columns = churn_df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_columns = churn_df.select_dtypes(include=["object", "category"]).columns.tolist()
print(f"Number of Numerical Columns   : {len(numerical_columns)}")
print(f"Number of Categorical Columns : {len(categorical_columns)}")

Number of Numerical Columns   : 3
Number of Categorical Columns : 18


In [22]:
display(churn_df.describe())

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [23]:
display(churn_df.describe(include=['object']).T)

,count,unique,top,freq
customerID,7043,7043,7590-VHVEG,1
gender,7043,2,Male,3555
Partner,7043,2,No,3641
Dependents,7043,2,No,4933
PhoneService,7043,2,Yes,6361
MultipleLines,7043,3,No,3390
InternetService,7043,3,Fiber optic,3096
OnlineSecurity,7043,3,No,3498
OnlineBackup,7043,3,No,3088
DeviceProtection,7043,3,No,3095


In [24]:
display(churn_df.describe(include="all"))

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
count,7043,7043,7043.000000,7043,7043,7043.000000,7043,7043,7043,7043,...,7043,7043,7043,7043,7043,7043,7043,7043.000000,7043,7043
unique,7043,2,NaN,2,2,NaN,2,3,3,3,...,3,3,3,3,3,2,4,NaN,6531,2
top,7590-VHVEG,Male,NaN,No,No,NaN,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,NaN,20.2,No
freq,1,3555,NaN,3641,4933,NaN,6361,3390,3096,3498,...,3095,3473,2810,2785,3875,4171,2365,NaN,11,5174
mean,NaN,NaN,0.162147,NaN,NaN,32.371149,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.761692,NaN,NaN
std,NaN,NaN,0.368612,NaN,NaN,24.559481,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.090047,NaN,NaN
min,NaN,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.250000,NaN,NaN
25%,NaN,NaN,0.000000,NaN,NaN,9.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.500000,NaN,NaN
50%,NaN,NaN,0.000000,NaN,NaN,29.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70.350000,NaN,NaN
75%,NaN,NaN,0.000000,NaN,NaN,55.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,89.850000,NaN,NaN


In [25]:
target_counts = churn_df["Churn"].value_counts()
target_counts

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [26]:
target_percentages = churn_df["Churn"].value_counts(normalize=True).mul(100).round(2)
target_percentages

Churn
No     73.46
Yes    26.54
Name: proportion, dtype: float64

In [27]:
target_distribution = pd.DataFrame({
    "Count": target_counts,
    "Percentage (%)": target_percentages
})

target_distribution

,Count,Percentage (%)
Churn,,
No,5174,73.46
Yes,1869,26.54


In [28]:
majority = target_counts.max()
minority = target_counts.min()
print("Majority = ",majority)
print("Minority = ",minority)

Majority =  5174
Minority =  1869


In [29]:
imbalance_ratio = round(majority / minority, 2)
print(f"Class Imbalance Ratio: {imbalance_ratio}:1")

Class Imbalance Ratio: 2.77:1


In [30]:
missing_summary = pd.DataFrame({
    "Missing Count": churn_df.isnull().sum(),
    "Missing Percentage (%)": (churn_df.isnull().mean()*100).round(2)})
display(missing_summary)

,Missing Count,Missing Percentage (%)
customerID,0,0.0
gender,0,0.0
SeniorCitizen,0,0.0
Partner,0,0.0
Dependents,0,0.0
tenure,0,0.0
PhoneService,0,0.0
MultipleLines,0,0.0
InternetService,0,0.0
OnlineSecurity,0,0.0


In [31]:
sns.countplot(x=churn_cleaned_df["Churn"])
plt.title("Count plot of target variable (churn)")
plt.xlabel("Target variable")
plt.ylabel("Count")
plt.show()

NameError: name 'churn_cleaned_df' is not defined